# Avro Basics — 01: Reading Avro Files

## What you will learn
Avro is the standard format for event streaming pipelines (Kafka, Pulsar).
Unlike Parquet (columnar/analytics), Avro is **row-based and schema-first**,
optimised for serialization and schema evolution.

In this notebook:
1. What Avro is and when to use it vs Parquet
2. Reading Avro files with `format("avro")`
3. Schema — how Avro embeds it in the file header
4. Read options — avroSchema, ignoreMissingFiles
5. Reading multiple Avro files
6. Verifying the spark-avro JAR is loaded


In [ ]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

In [ ]:
# Generate sample Avro files
import pathlib, random, datetime
random.seed(42)

AVRO_DIR = f"{DATA_DIR}/avro_files"
pathlib.Path(AVRO_DIR).mkdir(exist_ok=True)

from pyspark.sql.types import *

schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("event_type", StringType()),
    StructField("page",       StringType()),
    StructField("revenue",    DoubleType()),
    StructField("event_ts",   TimestampType()),
])

EVENTS = ["page_view","click","purchase","search","add_to_cart"]
PAGES  = ["/home","/products","/cart","/checkout"]

rows = []
base_ts = datetime.datetime(2024, 1, 1, 0, 0, 0)
for i in range(10_000):
    ts = base_ts + datetime.timedelta(seconds=i*10)
    rows.append((i+1, random.randint(1,500), random.choice(EVENTS),
                 random.choice(PAGES),
                 round(random.uniform(10,500),2) if random.random()>0.7 else 0.0,
                 ts))

df_events = spark.createDataFrame(rows, schema)
df_events.coalesce(2).write.format("avro").mode("overwrite").save(AVRO_DIR)
print(f"Written {df_events.count():,} events as Avro to {AVRO_DIR}")

import glob as glib
avro_files = glib.glob(f"{AVRO_DIR}/*.avro")
print(f"Files: {len(avro_files)}")
for f in avro_files:
    print(f"  {pathlib.Path(f).name}: {pathlib.Path(f).stat().st_size//1024} KB")

## Step 1 — Basic Read


In [ ]:
# Read Avro directory
df = spark.read.format("avro").load(AVRO_DIR)
print(f"Rows: {df.count():,}")
df.printSchema()
df.show(5)

## Step 2 — Avro Schema Inspection

Avro embeds its schema as JSON in the file header.
Let's read it with pyarrow to understand the format.


In [ ]:
try:
    import pyarrow as pa, pyarrow.dataset as ds
    dataset = ds.dataset(AVRO_DIR, format="avro")
    print("Avro schema (from file header):")
    print(dataset.schema)
except ImportError:
    pass

# Spark always reads the embedded schema automatically
print("\nSpark-inferred schema from Avro:")
df.printSchema()
print("Avro schema is embedded in file header — no schema argument needed for reading")

## Step 3 — avroSchema Option: Reader Schema


In [ ]:
# Read with explicit reader schema (subset of columns)
reader_schema = """
{
  "type": "record",
  "name": "Event",
  "fields": [
    {"name": "event_id",   "type": "long"},
    {"name": "user_id",    "type": "long"},
    {"name": "event_type", "type": "string"},
    {"name": "revenue",    "type": "double"}
  ]
}
"""

df_subset = spark.read.format("avro") \
                 .option("avroSchema", reader_schema) \
                 .load(AVRO_DIR)
print("Reading only 4 columns with explicit avroSchema:")
df_subset.show(5)
print(f"\nNote: Avro does NOT support column pruning like Parquet")
print("All bytes are read; only specified fields are returned to Spark")

## Step 4 — Reading Multiple Files


In [ ]:
# Multiple files as list
files = glib.glob(f"{AVRO_DIR}/*.avro")
df_multi = spark.read.format("avro").load(*files)
print(f"Read {len(files)} files: {df_multi.count():,} rows")

# Read with ignoreMissingFiles
df_safe = spark.read.format("avro") \
               .option("ignoreMissingFiles", "true") \
               .load(f"{AVRO_DIR}/*.avro",
                     f"{DATA_DIR}/nonexistent/*.avro")  # missing path ignored
print(f"With ignoreMissingFiles: {df_safe.count():,} rows (missing path ignored)")

# Read with recursiveFileLookup
pathlib.Path(f"{AVRO_DIR}/subdir").mkdir(exist_ok=True)
df_events.limit(100).coalesce(1).write.format("avro").mode("overwrite") \
         .save(f"{AVRO_DIR}/subdir")
df_recursive = spark.read.format("avro") \
                    .option("recursiveFileLookup", "true") \
                    .load(AVRO_DIR)
print(f"With recursiveFileLookup: {df_recursive.count():,} rows (includes subdir)")

## Summary

```python
# Basic read
spark.read.format("avro").load("/path/to/avro/")

# With reader schema (project columns)
spark.read.format("avro")
     .option("avroSchema", json_schema_string)
     .load(path)

# Options
spark.read.format("avro")
     .option("ignoreMissingFiles",    "true")
     .option("recursiveFileLookup",   "true")
     .option("pathGlobFilter",        "*.avro")
     .load(path)
```

### Avro vs Parquet for reading
| | Avro | Parquet |
|---|---|---|
| Column pruning | ❌ Reads all columns | ✅ Reads only needed columns |
| Predicate pushdown | ❌ No statistics | ✅ Min/max skipping |
| Schema in file | ✅ Header JSON | ✅ Footer binary |
| Best read use | Row-by-row processing | Analytical aggregations |
